# News Article Summarizer with ML
### Hex Softwares ML Internship - Project 1

**Intern:** Dhandabani M

**Models Used:**
- Abstractive: facebook/bart-large-cnn
- Extractive: TF-IDF + Sentence Scoring

**Features:**
- Paste text or fetch from URL
- Abstractive + Extractive summarization
- Keyword extraction and stats
- Gradio Web UI

In [8]:
!pip install transformers torch sentencepiece -q
!pip install gradio -q
!pip install newspaper3k -q
!pip install nltk -q
!pip install scikit-learn -q
!pip install lxml_html_clean -q
!pip install requests beautifulsoup4 -q
print('All packages installed!')

All packages installed!


In [9]:
import re
import string
import requests
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

from transformers import pipeline
from bs4 import BeautifulSoup

try:
    from newspaper import Article
    NEWSPAPER_AVAILABLE = True
except:
    NEWSPAPER_AVAILABLE = False

import gradio as gr

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
print('All libraries imported!')

All libraries imported!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [21]:
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

print('Loading BART model... (1-2 minutes on first run)')

MODEL_NAME = 'facebook/bart-large-cnn'

# Determine the device dynamically
# Using -1 for CPU, or an integer for GPU index (0 for the first GPU)
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'cuda:0' if device == 0 else 'cpu'}")

# Explicitly load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device) # Move model to determined device

# Define a function for abstractive summarization using the model and tokenizer directly
def generate_abstractive_summary(text, max_length, min_length):
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()} # Move inputs to model device

    summary_ids = model.generate(
        inputs['input_ids'],
        num_beams=4,
        max_length=max_length,
        min_length=min_length,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f'Model and tokenizer loaded for direct summarization: {MODEL_NAME}')

Loading BART model... (1-2 minutes on first run)
Using device: cuda:0


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Model and tokenizer loaded for direct summarization: facebook/bart-large-cnn


In [14]:
def extractive_summarize(text, num_sentences=5):
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    stop_words = set(stopwords.words('english'))
    clean_sentences = []
    for sent in sentences:
        words = word_tokenize(sent.lower())
        words = [w for w in words if w not in stop_words and w not in string.punctuation]
        clean_sentences.append(' '.join(words))
    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(clean_sentences)
    except:
        return ' '.join(sentences[:num_sentences])
    sentence_scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
    top_indices = sorted(np.argsort(sentence_scores)[-num_sentences:])
    return ' '.join([sentences[i] for i in top_indices])

print('Extractive summarizer ready!')

Extractive summarizer ready!


In [15]:
def fetch_article_from_url(url):
    if NEWSPAPER_AVAILABLE:
        try:
            article = Article(url)
            article.download()
            article.parse()
            if len(article.text) > 100:
                return article.title, article.text, True
        except Exception as e:
            print(f'newspaper3k failed: {e}')
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        title = soup.find('h1')
        title_text = title.get_text(strip=True) if title else 'No title'
        paragraphs = soup.find_all('p')
        article_text = ' '.join([p.get_text(strip=True) for p in paragraphs])
        if len(article_text) > 100:
            return title_text, article_text, True
        return '', '', False
    except Exception as e:
        return '', f'Error: {str(e)}', False

print('URL fetcher ready!')

URL fetcher ready!


In [16]:
def extract_keywords(text, top_n=10):
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text.lower())
    words = [w for w in words if w.isalpha() and w not in stop_words and len(w) > 3]
    freq_dist = FreqDist(words)
    return [word for word, freq in freq_dist.most_common(top_n)]

def get_text_stats(original_text, summary_text):
    orig_words = len(word_tokenize(original_text))
    summ_words = len(word_tokenize(summary_text))
    orig_sents = len(sent_tokenize(original_text))
    summ_sents = len(sent_tokenize(summary_text))
    compression = round((1 - summ_words / orig_words) * 100, 1) if orig_words > 0 else 0
    return (
        f'Original : {orig_words} words | {orig_sents} sentences\n'
        f'Summary  : {summ_words} words | {summ_sents} sentences\n'
        f'Compression: {compression}% reduction'
    )

print('Keywords and stats functions ready!')

Keywords and stats functions ready!


In [17]:
def summarize_text(text, method='abstractive', max_length=150, min_length=50, num_sentences=5):
    if not text or len(text.strip()) < 50:
        return {'error': 'Please provide at least 50 characters.'}
    text = re.sub(r'\s+', ' ', text).strip()
    results = {}
    if method in ['abstractive', 'both']:
        try:
            output = abstractive_summarizer(
                text[:3000],
                max_length=max_length,
                min_length=min_length,
                do_sample=False,
                truncation=True
            )
            results['abstractive'] = output[0]['summary_text']
        except Exception as e:
            results['abstractive'] = f'Error: {str(e)}'
    if method in ['extractive', 'both']:
        try:
            results['extractive'] = extractive_summarize(text, num_sentences=num_sentences)
        except Exception as e:
            results['extractive'] = f'Error: {str(e)}'
    primary = results.get('abstractive') or results.get('extractive', '')
    results['keywords'] = extract_keywords(text)
    results['stats'] = get_text_stats(text, primary)
    return results

print('Main summarize function ready!')

Main summarize function ready!


In [18]:
sample = (
    'Tesla announced a major breakthrough in electric vehicle battery technology. '
    'The new 4680 battery cells will reduce the cost of electric vehicles '
    'and extend range by up to 16 percent. CEO Elon Musk unveiled the design at '
    'Battery Day calling it a fundamental rethinking of battery design. '
    'The cells use a tabless design that improves energy flow and reduces heat. '
    'Tesla claims five times more energy storage and six times more power output. '
    'Battery costs are estimated to drop by 56 percent as production scales up. '
    'Industry analysts praised the announcement but noted manufacturing challenges remain. '
    'Tesla remains the worlds most valuable automaker by market capitalization.'
)

print('--- Abstractive Summary ---')
r1 = summarize_text(sample, method='abstractive', max_length=100, min_length=30)
print(r1.get('abstractive', ''))

print('\n--- Extractive Summary ---')
r2 = summarize_text(sample, method='extractive', num_sentences=3)
print(r2.get('extractive', ''))

print('\n--- Keywords ---')
print(', '.join(r1.get('keywords', [])))

print('\n--- Stats ---')
print(r1.get('stats', ''))

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'min_length', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


--- Abstractive Summary ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Error: 'summary_text'

--- Extractive Summary ---
The new 4680 battery cells will reduce the cost of electric vehicles and extend range by up to 16 percent. CEO Elon Musk unveiled the design at Battery Day calling it a fundamental rethinking of battery design. The cells use a tabless design that improves energy flow and reduces heat.

--- Keywords ---
battery, tesla, design, electric, cells, percent, energy, times, announced, major

--- Stats ---
Original : 114 words | 8 sentences
Summary  : 4 words | 1 sentences
Compression: 96.5% reduction


In [19]:
def gradio_summarize(input_type, article_text, article_url,
                     method, max_length, min_length, num_sentences):
    if input_type == 'Paste Text':
        text = article_text
        title = 'Custom Article'
    else:
        if not article_url or not article_url.startswith('http'):
            return 'Please enter a valid URL.', '', '', '', ''
        title, text, success = fetch_article_from_url(article_url)
        if not success:
            return 'Could not fetch article. Paste text directly.', '', '', '', ''
    if not text or len(text.strip()) < 100:
        return 'No text found. Provide more content.', '', '', '', ''
    method_map = {
        'Abstractive (BART)': 'abstractive',
        'Extractive (TF-IDF)': 'extractive',
        'Both Methods': 'both'
    }
    results = summarize_text(
        text, method=method_map.get(method, 'abstractive'),
        max_length=int(max_length), min_length=int(min_length),
        num_sentences=int(num_sentences)
    )
    if 'error' in results:
        return results['error'], '', '', '', ''
    return (
        f'Done! Article: {title[:60]}',
        results.get('abstractive', 'Not selected'),
        results.get('extractive', 'Not selected'),
        'Keywords: ' + ' | '.join(results.get('keywords', [])),
        results.get('stats', '')
    )


with gr.Blocks(title='News Article Summarizer',
               theme=gr.themes.Soft(primary_hue='blue')) as demo:

    gr.Markdown('# News Article Summarizer')
    gr.Markdown('**Hex Softwares ML Internship - Project 1 | Dhandabani M**')

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Input')
            input_type = gr.Radio(
                choices=['Paste Text', 'Fetch from URL'],
                value='Paste Text', label='Input Method'
            )
            article_text = gr.Textbox(
                label='Paste News Article',
                placeholder='Paste your news article text here...',
                lines=10
            )
            article_url = gr.Textbox(
                label='Or Enter News URL',
                placeholder='https://www.bbc.com/news/...',
                visible=False
            )
            def toggle_input(choice):
                if choice == 'Fetch from URL':
                    return gr.update(visible=False), gr.update(visible=True)
                return gr.update(visible=True), gr.update(visible=False)
            input_type.change(toggle_input, inputs=input_type,
                              outputs=[article_text, article_url])
            gr.Markdown('### Settings')
            method = gr.Radio(
                choices=['Abstractive (BART)', 'Extractive (TF-IDF)', 'Both Methods'],
                value='Abstractive (BART)', label='Method'
            )
            with gr.Row():
                max_len = gr.Slider(50, 300, value=150, step=10, label='Max Length')
                min_len = gr.Slider(20, 100, value=50, step=5, label='Min Length')
            num_sents = gr.Slider(1, 10, value=5, step=1, label='Num Sentences (Extractive)')
            btn = gr.Button('Summarize', variant='primary', size='lg')

        with gr.Column(scale=1):
            gr.Markdown('### Results')
            status_out = gr.Textbox(label='Status', interactive=False)
            abs_out = gr.Textbox(label='Abstractive Summary (BART)', lines=5, interactive=False)
            ext_out = gr.Textbox(label='Extractive Summary (TF-IDF)', lines=5, interactive=False)
            kw_out = gr.Textbox(label='Top Keywords', lines=2, interactive=False)
            stats_out = gr.Textbox(label='Statistics', lines=5, interactive=False)

    btn.click(
        fn=gradio_summarize,
        inputs=[input_type, article_text, article_url, method, max_len, min_len, num_sents],
        outputs=[status_out, abs_out, ext_out, kw_out, stats_out]
    )

print('Launching Gradio App...')
demo.launch(share=True, debug=False)

Launching Gradio App...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f63d60ca9cf1a890cc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
